In [1]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('titanic.csv')

In [6]:
print("Initial shape:", df.shape)
print("Missing values:\n", df.isnull().sum())


Initial shape: (891, 12)
Missing values:
 PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [7]:
# For 'Age': fill with median (robust to outliers)
df['Age'].fillna(df['Age'].median(), inplace=True)

# For 'Embarked': fill with mode (most frequent port)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# For 'Cabin': drop column (too many missing values, low predictive value)
df.drop('Cabin', axis=1, inplace=True)

# For 'Fare': fill with median (only if missing – usually Titanic dataset has none)
df['Fare'].fillna(df['Fare'].median(), inplace=True)

/tmp/ipykernel_12355/4081507730.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
/tmp/ipykernel_12355/4081507730.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', 

In [8]:
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - df.shape[0]} duplicate rows")

Removed 0 duplicate rows


In [9]:
def cap_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return series.clip(lower, upper)

numeric_cols = ['Age', 'Fare', 'SibSp', 'Parch']
for col in numeric_cols:
    df[col] = cap_outliers_iqr(df[col])

In [10]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[numeric_cols] = scaler.fit_transform(df[numeric_cols])

In [11]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
df = pd.get_dummies(df, columns=['Embarked'], prefix='Emb', drop_first=True)

In [12]:
print("\nFinal shape:", df.shape)
print("Remaining missing values:\n", df.isnull().sum().sum())   # should be 0

# Save cleaned data
df.to_csv('titanic_cleaned.csv', index=False)
print("Cleaned dataset saved as 'titanic_cleaned.csv'")


Final shape: (891, 12)
Remaining missing values:
 0
Cleaned dataset saved as 'titanic_cleaned.csv'
